# Experiment 2 — Per-patient evaluation + bootstrap 95% CI

Notebook này **không train lại model**. Nó đọc OOF predictions từ notebook 5-fold CV hiện tại.

Mục tiêu:
1. tính metric voxel-level **riêng cho từng patient_id**;
2. tổng hợp mean ± SD, median và IQR;
3. bootstrap theo **patient_id**, không bootstrap từng voxel, để tạo 95% CI cho trung bình metric theo bệnh nhân.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, balanced_accuracy_score
)

RANDOM_STATE = 42
N_BOOT = 5000
THRESHOLD = 0.50

# OOF chính thức do Experiment 1 xuất ra.
OOF_PATH = "output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv"
OUTPUT_DIR = "output/577p/glcm/kernel5/per_patient_bootstrap_ci"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(OOF_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy OOF: {OOF_PATH}\n"
        "Hãy chạy notebook 01_XGBoost_GLCM_ClassWeight_Ablation_FIXED.ipynb trước."
    )

print("OOF source:", OOF_PATH)


OOF source: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv


## 1. Load OOF predictions

In [2]:
oof = pd.read_csv(OOF_PATH)

PATIENT_CANDIDATES = ["patient_id", "patient", "subject_id", "case_id"]
patient_col = next((c for c in PATIENT_CANDIDATES if c in oof.columns), None)

required = {"label", "prob_tumor"}
if patient_col is None:
    raise ValueError("OOF file thiếu patient_id.")
if not required.issubset(oof.columns):
    raise ValueError(f"OOF file thiếu cột: {required - set(oof.columns)}")

oof[patient_col] = oof[patient_col].astype(str)
oof["label"] = oof["label"].astype(int)

print("OOF rows:", len(oof))
print("Patients:", oof[patient_col].nunique())
print(oof["label"].value_counts().sort_index())

OOF rows: 653397
Patients: 577
0    611803
1     41594
Name: label, dtype: int64


## 2. Per-patient metrics

In [3]:
def binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

In [4]:
patient_rows = []

for pid, g in oof.groupby(patient_col, sort=True):
    y = g["label"].to_numpy(dtype=int)
    p = g["prob_tumor"].to_numpy(dtype=float)
    m = binary_metrics(y, p, threshold=THRESHOLD)

    patient_rows.append({
        patient_col: pid,
        "n_voxels": len(g),
        "n_non_tumor": int((y == 0).sum()),
        "n_tumor": int((y == 1).sum()),
        "n_classes": int(np.unique(y).size),
        **m
    })

patient_metrics = pd.DataFrame(patient_rows)
patient_metrics.to_csv(os.path.join(OUTPUT_DIR, "per_patient_metrics.csv"), index=False)

print("Patients with both classes:", int((patient_metrics["n_classes"] == 2).sum()), "/", len(patient_metrics))
display(patient_metrics.head())

Patients with both classes: 577 / 577


,patient_id,n_voxels,n_non_tumor,n_tumor,n_classes,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,BraTS2021_00000,1112,1069,43,2,0.982914,0.801390,0.928571,0.604651,0.998129,0.732394,0.741803,0.980573,0.829537,1067,2,17,26
1,BraTS2021_00002,1054,913,141,2,0.929791,0.830541,0.759690,0.695035,0.966046,0.725926,0.686648,0.968050,0.842524,882,31,43,98
2,BraTS2021_00003,1214,1142,72,2,0.895387,0.937889,0.360406,0.986111,0.889667,0.527881,0.561048,0.996060,0.959924,1016,126,1,71
3,BraTS2021_00005,961,866,95,2,0.963580,0.975106,0.734375,0.989474,0.960739,0.843049,0.834669,0.993692,0.952283,832,34,1,94
4,BraTS2021_00006,1131,1027,104,2,0.905393,0.943586,0.492823,0.990385,0.896787,0.658147,0.660497,0.995403,0.978933,921,106,1,103


## 3. Descriptive summary

In [5]:
metric_cols = [
    "accuracy","balanced_accuracy","precision","recall","specificity",
    "f1","mcc","roc_auc","pr_auc"
]

summary_rows = []
for metric in metric_cols:
    x = patient_metrics[metric].dropna().astype(float)
    summary_rows.append({
        "metric": metric,
        "n_patients_valid": len(x),
        "mean": x.mean(),
        "sd": x.std(ddof=1),
        "median": x.median(),
        "q1": x.quantile(0.25),
        "q3": x.quantile(0.75)
    })

descriptive = pd.DataFrame(summary_rows)
descriptive.to_csv(os.path.join(OUTPUT_DIR, "per_patient_descriptive_summary.csv"), index=False)
display(descriptive)

,metric,n_patients_valid,mean,sd,median,q1,q3
0,accuracy,577,0.915410,0.118613,0.959821,0.906126,0.980312
1,balanced_accuracy,577,0.903300,0.083692,0.930976,0.862973,0.964985
2,precision,577,0.577045,0.244181,0.640000,0.407407,0.764977
3,recall,577,0.891406,0.147689,0.950000,0.860465,0.987013
4,specificity,577,0.915194,0.130973,0.967141,0.904891,0.985445
5,f1,577,0.653603,0.209461,0.708661,0.542373,0.815789
6,mcc,577,0.659769,0.193283,0.704572,0.554100,0.808918
7,roc_auc,577,0.980944,0.027566,0.991330,0.976606,0.996383
8,pr_auc,577,0.858603,0.136600,0.902311,0.822099,0.944458


## 4. Bootstrap 95% CI theo patient_id

Mỗi bootstrap replicate lấy mẫu **bệnh nhân với hoàn lại** rồi tính **trung bình metric theo bệnh nhân**.
Vì vậy bệnh nhân, không phải voxel, là đơn vị bootstrap.

In [6]:
rng = np.random.default_rng(RANDOM_STATE)
n_patients = len(patient_metrics)

boot_rows = []
for b in range(N_BOOT):
    idx = rng.integers(0, n_patients, size=n_patients)
    sample = patient_metrics.iloc[idx]

    row = {"bootstrap": b + 1}
    for metric in metric_cols:
        row[metric] = sample[metric].dropna().mean()
    boot_rows.append(row)

boot = pd.DataFrame(boot_rows)
boot.to_csv(os.path.join(OUTPUT_DIR, "patient_bootstrap_replicates.csv"), index=False)

ci_rows = []
for metric in metric_cols:
    x = boot[metric].dropna().to_numpy(dtype=float)
    ci_rows.append({
        "metric": metric,
        "bootstrap_n": len(x),
        "point_estimate_mean_patient_metric": patient_metrics[metric].dropna().mean(),
        "ci95_low_percentile": np.quantile(x, 0.025),
        "ci95_high_percentile": np.quantile(x, 0.975)
    })

ci = pd.DataFrame(ci_rows)
ci.to_csv(os.path.join(OUTPUT_DIR, "per_patient_bootstrap_95ci.csv"), index=False)
display(ci)

,metric,bootstrap_n,point_estimate_mean_patient_metric,ci95_low_percentile,ci95_high_percentile
0,accuracy,5000,0.915410,0.905515,0.924950
1,balanced_accuracy,5000,0.903300,0.896540,0.910155
2,precision,5000,0.577045,0.556665,0.596802
3,recall,5000,0.891406,0.879061,0.902834
4,specificity,5000,0.915194,0.904179,0.925734
5,f1,5000,0.653603,0.636116,0.670798
6,mcc,5000,0.659769,0.643745,0.675665
7,roc_auc,5000,0.980944,0.978578,0.983149
8,pr_auc,5000,0.858603,0.847149,0.869438


## Reporting note

Các CI này là **CI của trung bình metric voxel-level tính theo từng bệnh nhân**.
Không gọi chúng là patient-level disease classification metrics.